In [ ]:
## Modeling approach

In this notebook, we explore modeling approach with NGBoost. Our dataset has the following structure:

| FSA | features | concentration |
| :--- | :---: | ---: |
| A | features(A) | $r_{A,1}$ |
| $\vdots$ | $\vdots$ | $\vdots$ |
| A | features(A) | $r_{A,n_A}$ |
| B | features(B) | $r_{B,1}$ |
| $\vdots$ | $\vdots$| $\vdots$ |
| B | features(B) | $r_{B,n_B}$ |
| $\vdots$ | $\vdots$ | $\vdots$ |

Here features(A) denotes all the features columns of that particular FSA (which remains the same per FSA), and $r_{A, i}$ denotes the radon concentration level of $i$-th measurement in the FSA A.

The idea is that given enough measurements in an FSA ..................................

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import geopandas as gpd
import sys
import matplotlib.pyplot as plt
# ---------------------------------------------------------------------
# Add project root to Python path
# ---------------------------------------------------------------------


# project root = two levels above notebooks
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
## Helper Functions
from src.modeling.utils import fsa_grouped_features_and_proportion_above200
def extract_X_y(df, is_train, approach = "Naive",):
    if is_train == False:
        df = fsa_grouped_features_and_proportion_above200(df)
        y = df['y_mean']
    else:
        y = df['concentration']
    if approach == "Naive":
        X = df.drop(columns=['concentration', 'provinceterritory', 'FSA', 'geometry'], 
                    errors = 'ignore')
    elif approach == "highly_correlated_only":
        columns_to_keep = ['hous_frac_type_single_detached', 'mean_uranium', 'geolprov_interior_platform',
                           'socioeco_frac_housing_burden', 'hous_median_value', 'hous_frac_type_other_attached']
        columns_to_drop = df.columns.difference(columns_to_keep)
        X = df.drop(columns= columns_to_drop)
        
    return (X,y)


In [ ]:
from src.data.data_loading import load_training_data,load_validation_data
remove_columns = ['spatial_cluster', 'is_test', 'cv_fold']
train_df = load_training_data(cv_fold=0).drop(columns= remove_columns)
val_df = load_validation_data(cv_fold=0).drop(columns= remove_columns)

In [ ]:
from ngboost import NGBRegressor
from ngboost.distns import LogNormal

X_train, y_train = extract_X_y(train_df, is_train= True)
X_val, y_val = extract_X_y(val_df, is_train= False)

ngb = NGBRegressor(Dist=LogNormal).fit(X_train, y_train)

dist = ngb.pred_dist(X_val)

# Get the probability that radon > 200 for each FSA
# 1 - CDF(200) gives the "exceedance probability"
probs_above_200 = 1 - dist.cdf(200)

In [ ]:
from src.modeling.utils import fsa_proportion_above200
from sklearn.metrics import mean_absolute_error
y_true = fsa_proportion_above200(val_df)['proportion'].tolist()

mean_absolute_error(y_true=y_true, y_pred= probs_above_200)

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_true, probs_above_200, alpha=0.5, color='purple')
plt.plot([0, 1], [0, 1], color='black', linestyle='--') # Perfect calibration line
plt.xlabel("Actual Proportion of Homes > 200")
plt.ylabel("Predicted Probability of Home > 200")
plt.title("Calibration Plot: Predicted Risk vs. Observed Reality")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from src.modeling.utils import no_of_measurements_per_fsa

fsas = no_of_measurements_per_fsa(val_df)
fsas_sorted = no_of_measurements_per_fsa(val_df).sort_values('no_of_measurements')
idx = fsas.index[fsas['FSA'].isin(fsas_sorted['FSA'].tolist()[-3:-1])]

fsa_names = fsas['FSA'].tolist()
## Plotting distribution for the FSAs with most number of observations
idxs = [int(idx[i]) for i in range(len(idx))]
#idxs = [len_val_fsas-1]
x_range = np.linspace(0, 600, 500)

for idx in idxs:
    # Get the PDF for this specific instance
    pdf_values = dist[idx].pdf(x_range)
    max_pdf_value = max(pdf_values)
    plt.figure(figsize=(10, 5))
    plt.plot(x_range, pdf_values, label=f"FSA: {fsa_names[idx]}", color='teal')
    plt.fill_between(x_range, pdf_values, where=(x_range > 200), color='red', alpha=0.3, label='Risk Area (>200)')
    plt.axvline(200, color='black', linestyle='--', label='Action Level (200)')
    plt.text(0.79*600, 0.7*max_pdf_value, f"Probability of Risk: {1 - dist[idx].cdf(200):.2f}", color = 'red')
    plt.text(0.65*600, 0.65*max_pdf_value, f"Obsereved Proportion above Risk: {y_true[idx]:.2f}", color = 'red')
    plt.title(f"Radon Probability Distribution for FSA {fsa_names[idx]}")
    plt.xlabel("Radon Concentration (Bq/m³)")
    plt.ylabel("Probability Density")
    plt.legend()
    plt.show()